# De-identifying clinical free text: worked solution

Worked solution for the exercise in the lesson **[De-identifying clinical free text](https://ramkumarmd.com/medical-ai/clinical-nlp/de-identifying-clinical-text)** (*Clinical NLP* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](04-de-identifying-clinical-text.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** De-identifying clinical notes means finding and removing or replacing names, dates, record numbers, contact details, addresses and other identifiers in free text. Rule-based patterns catch structured identifiers such as phone numbers and dates, but miss names without titles, places and unusual details; machine-learning tools do better but are not perfect. Because a single missed identifier can expose a patient, measure recall carefully, combine methods, replace identifiers with realistic surrogates, and keep de-identified text under governance.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### A rule-based detector

Patterns for titled names, record numbers, dates, phone numbers, emails and postcodes, on a synthetic note.

In [ ]:
import re

# Synthetic note: every name, number and place below is invented
note = ("Mrs Asha Verma (MRN 00482913), DOB 04/02/1958, seen on 12/03/2026 at Lakeside Clinic. "
        "Accompanied by her son Ravi. Phone 98765 43210, email asha.verma@example.org. "
        "Dr Lee reviewed her foot ulcer. Lives at 14 Hill Road, PIN 999999. Retired school principal.")

# The identifiers a careful human reviewer marked in this note
GOLD = ["Asha Verma", "00482913", "04/02/1958", "12/03/2026", "Lakeside Clinic", "Ravi",
        "98765 43210", "asha.verma@example.org", "Lee", "14 Hill Road", "999999"]

PATTERNS = [
    ("NAME", r"\b(?:Mr|Mrs|Ms|Dr)\.?\s+((?:[A-Z][a-z]+)(?:\s+[A-Z][a-z]+)?)"),
    ("ID", r"\bMRN\s*(\d{6,})"),
    ("DATE", r"\b(\d{1,2}/\d{1,2}/\d{2,4})\b"),
    ("PHONE", r"\b(\d{5}\s?\d{5})\b"),
    ("EMAIL", r"\b([\w.+-]+@[\w-]+\.[\w.]+)\b"),
    ("POSTCODE", r"\bPIN\s*(\d{6})\b"),
]

def find_phi(text):
    spans = []
    for label, pattern in PATTERNS:
        for m in re.finditer(pattern, text):
            spans.append((m.start(1), m.end(1), label, m.group(1)))
    return sorted(spans)

found = [s[3] for s in find_phi(note)]
missed = [g for g in GOLD if not any(g in f or f in g for f in found)]
print(f"Found {len(GOLD) - len(missed)} of {len(GOLD)} identifiers (recall {1 - len(missed) / len(GOLD):.0%})")
print("Missed:", missed)

# Found 8 of 11 identifiers (recall 73%)
# Missed: ['Lakeside Clinic', 'Ravi', '14 Hill Road']

### Replacing identifiers: tags and surrogates

Continuing from the previous block: tag identifiers and shift dates consistently.

In [ ]:
from datetime import datetime, timedelta

SHIFT = timedelta(days=-37)        # one random offset per patient, kept secret
def surrogate(label, value):
    if label == "DATE":
        d = datetime.strptime(value, "%d/%m/%Y") + SHIFT
        return d.strftime("%d/%m/%Y")
    return f"[{label}]"

def redact(text):
    for start, end, label, value in reversed(find_phi(text)):
        text = text[:start] + surrogate(label, value) + text[end:]
    return text

print(redact(note))

# Mrs [NAME] (MRN [ID]), DOB 29/12/1957, seen on 03/02/2026 at Lakeside Clinic. Accompanied by her son Ravi. Phone [PHONE], email [EMAIL]. Dr [NAME] reviewed her foot ulcer. Lives at 14 Hill Road, PIN [POSTCODE]. Retired school principal.

## Exercise data

Synthetic data written for this exercise (no real patients). Run this cell before starting the steps.

In [ ]:
# Fifteen SYNTHETIC notes with the identifiers a careful reviewer marked, by type. Every name, number, address and
# place is invented (phone numbers start 00000, emails use example.org, postcodes are 9s). Notes 1-3 and 4-5 belong
# to the same two patients, for the consistent-surrogate step.
DEID_NOTES = [
    ("A", "Mrs Asha Verma (MRN 00482913), DOB 04/02/1958, seen on 12/03/2026 at Lakeside Clinic. Accompanied by her son Ravi.",
     {"NAME": ["Asha Verma", "Ravi"], "ID": ["00482913"], "DATE": ["04/02/1958", "12/03/2026"], "FACILITY": ["Lakeside Clinic"]}),
    ("A", "Asha Verma reviewed on 09/04/2026. Her son Ravi says her feet are sore. Phone 00000 43210.",
     {"NAME": ["Asha Verma", "Ravi"], "DATE": ["09/04/2026"], "PHONE": ["00000 43210"]}),
    ("A", "Mrs Verma lives at 14 Hill Road, PIN 999999. Email asha.verma@example.org. Seen by Dr Lee.",
     {"NAME": ["Verma", "Lee"], "ADDRESS": ["14 Hill Road"], "POSTCODE": ["999999"], "EMAIL": ["asha.verma@example.org"]}),
    ("B", "Mr John Mathew, MRN 00918273, admitted to Riverside Hospital on 02/05/2026 with a foot infection.",
     {"NAME": ["John Mathew"], "ID": ["00918273"], "FACILITY": ["Riverside Hospital"], "DATE": ["02/05/2026"]}),
    ("B", "John Mathew discharged 09/05/2026. His wife Mary will help with dressings. Call 00000 55512.",
     {"NAME": ["John Mathew", "Mary"], "DATE": ["09/05/2026"], "PHONE": ["00000 55512"]}),
    ("C", "Ms Priya Rao, DOB 11/11/1990, gestational diabetes, reviewed at Greenfield Clinic.",
     {"NAME": ["Priya Rao"], "DATE": ["11/11/1990"], "FACILITY": ["Greenfield Clinic"]}),
    ("D", "Dr Okafor discussed results with Mr Sanjay Gupta by phone on 15/06/2026.",
     {"NAME": ["Okafor", "Sanjay Gupta"], "DATE": ["15/06/2026"]}),
    ("E", "Patient Leela Nair (MRN 00733419) lives at 7 Lake View Street, PIN 999998.",
     {"NAME": ["Leela Nair"], "ID": ["00733419"], "ADDRESS": ["7 Lake View Street"], "POSTCODE": ["999998"]}),
    ("F", "Mrs Fatima Khan attended with her daughter Sara. Email f.khan@example.org.",
     {"NAME": ["Fatima Khan", "Sara"], "EMAIL": ["f.khan@example.org"]}),
    ("G", "Mr Peter Brown, retired bus driver, seen 21/07/2026 at Northgate Hospital.",
     {"NAME": ["Peter Brown"], "DATE": ["21/07/2026"], "FACILITY": ["Northgate Hospital"]}),
    ("H", "Kiran Reddy, 34, phone 00000 77788, works at the city library.",
     {"NAME": ["Kiran Reddy"], "PHONE": ["00000 77788"]}),
    ("I", "Mrs Grace Thomas, MRN 00651200. Her husband George manages her insulin.",
     {"NAME": ["Grace Thomas", "George"], "ID": ["00651200"]}),
    ("J", "Seen at Hilltop Clinic on 03/08/2026. Mr Arun Das lives at 22 Station Road.",
     {"FACILITY": ["Hilltop Clinic"], "DATE": ["03/08/2026"], "NAME": ["Arun Das"], "ADDRESS": ["22 Station Road"]}),
    ("K", "Ms Neha Joshi, email neha.j@example.org, DOB 30/01/1985, type 1 diabetes.",
     {"NAME": ["Neha Joshi"], "EMAIL": ["neha.j@example.org"], "DATE": ["30/01/1985"]}),
    ("L", "Mr Ali Hassan reviewed by Dr Mehta. His son Omar drove him to Central Hospital.",
     {"NAME": ["Ali Hassan", "Mehta", "Omar"], "FACILITY": ["Central Hospital"]}),
]
print(f"{len(DEID_NOTES)} synthetic notes, {sum(len(v) for _, _, g in DEID_NOTES for v in g.values())} marked identifiers")

## Exercise

**Goal:** Build, evaluate and improve a de-identifier on synthetic notes.

**Data:** Fifteen synthetic notes with marked identifiers, written for the notebook (Created for this course; no real patient data)

### Step 1

Reproduce the detector on all fifteen notes and report recall by identifier type.

In [ ]:
import pandas as pd
from collections import defaultdict

def recall_by_type(find, notes=DEID_NOTES):
    hits, totals = defaultdict(int), defaultdict(int)
    found_strings_all, false_positives = [], 0
    for _, text, gold in notes:
        found = [s[3] for s in find(text)]
        gold_values = [v for vs in gold.values() for v in vs]
        for kind, values in gold.items():
            for v in values:
                totals[kind] += 1
                hits[kind] += any(v in f or f in v for f in found)
        false_positives += sum(not any(f in g or g in f for g in gold_values) for f in found)
    table = pd.DataFrame({"marked": totals, "found": hits}).fillna(0).astype(int)
    table["recall"] = (table.found / table.marked).round(2)
    print(table.sort_values("recall").to_string())
    print(f"overall recall {table.found.sum() / table.marked.sum():.0%}; false positives {false_positives}")
    return table

baseline = recall_by_type(find_phi)

### Step 2

Add patterns for addresses, untitled names after relationship words ('her son', 'his wife') and facility names ending in 'Clinic' or 'Hospital'; re-measure recall and count false positives.

In [ ]:
PATTERNS_V2 = PATTERNS + [
    ("ADDRESS", r"\b(\d{1,4}\s+(?:[A-Z][a-z]+\s+){1,3}(?:Road|Street|Lane|Avenue|Nagar))\b"),
    ("NAME", r"\b(?:[Hh]is|[Hh]er|[Tt]heir)\s+(?:son|daughter|wife|husband|brother|sister|mother|father)\s+([A-Z][a-z]+)"),
    ("FACILITY", r"\b((?:[A-Z][a-z]+\s+){1,2}(?:Clinic|Hospital))\b"),
    ("NAME", r"\b(?:Patient|Pt)\s+([A-Z][a-z]+\s+[A-Z][a-z]+)"),
    ("NAME", r"^([A-Z][a-z]+\s+[A-Z][a-z]+)(?=\s+(?:reviewed|discharged|attended|,))"),
    ("ID", r"\bMRN[:\s]*,?\s*(\d{6,})"),
]

def find_phi_v2(text):
    spans = []
    for label, pattern in PATTERNS_V2:
        for m in re.finditer(pattern, text, flags=re.MULTILINE):
            spans.append((m.start(1), m.end(1), label, m.group(1)))
    # remove spans contained in a longer span
    spans = sorted(set(spans), key=lambda s: (s[0], -s[1]))
    kept = [s for s in spans if not any(o[0] <= s[0] and s[1] <= o[1] and o != s for o in spans)]
    return kept

improved = recall_by_type(find_phi_v2)

The new patterns find addresses, relatives named after relationship words, facility names and untitled names at the
start of a note. Watch the false-positive count: "Central Hospital" is right, but a pattern for "two capitalised
words" would also catch "Type One" or "Lake View"; precision matters because over-redaction destroys clinical meaning.
What is still missed (look at the table) usually needs a statistical name recogniser rather than another rule.

### Step 3

Replace tags with realistic surrogate names and check that names are replaced consistently across a patient's notes.

In [ ]:
import hashlib

FIRST = ["Anita", "Bala", "Chen", "Deepa", "Elena", "Farid", "Gita", "Hari", "Ines", "Jamal", "Kavya", "Liam"]
LAST = ["Iyer", "Smith", "Okoro", "Sato", "Kumar", "Silva", "Novak", "Haddad", "Menon", "Wright"]

def surrogate_name(original, patient_key):
    """Same original name (within one patient's notes) -> same made-up name; different patients get different maps."""
    h = int(hashlib.sha256(f"{patient_key}|{original}".encode()).hexdigest(), 16)
    parts = original.split()
    return FIRST[h % len(FIRST)] + (f" {LAST[(h // 7) % len(LAST)]}" if len(parts) > 1 else "")

def pseudonymise(text, patient_key):
    for start, end, label, value in sorted(find_phi_v2(text), reverse=True):
        if label == "NAME":
            replacement = surrogate_name(value, patient_key)
        elif label == "DATE":
            replacement = (datetime.strptime(value, "%d/%m/%Y") + SHIFT).strftime("%d/%m/%Y")
        else:
            replacement = f"[{label}]"
        text = text[:start] + replacement + text[end:]
    return text

for patient, text, _ in DEID_NOTES[:5]:
    print(f"{patient}: {pseudonymise(text, patient)}")

Consistency works where the same string recurs: "Asha Verma" and "Ravi" get the same surrogates in each of patient
A's notes, so the notes still read as one story. The output also shows three problems a reviewer must catch:

- **Partial names**: "Mrs Verma" (surname only) is replaced by a different surrogate from "Asha Verma"; names need
  normalising (linking surname-only mentions to the full name) before surrogates are assigned.
- **Gender and type**: the surrogate list ignores gender, so "Mr John Mathew" and "his wife Mary" get names that do
  not fit, and a surname ("Dr Lee") can become a first name. Real surrogate generators keep gender, name part and,
  ideally, cultural plausibility.
- **Secrecy**: the mapping key (here, the patient key plus a fixed hash) must be kept secret, or the surrogates can be
  reversed by trial.

### Step 4

Optional: run Microsoft Presidio on the same notes and compare.

Optional (outline, not run here: Presidio needs a spaCy language model of several hundred megabytes).

```python
# pip install presidio-analyzer presidio-anonymizer && python -m spacy download en_core_web_lg
from presidio_analyzer import AnalyzerEngine
analyzer = AnalyzerEngine()
def find_phi_presidio(text):
    return [(r.start, r.end, r.entity_type, text[r.start:r.end])
            for r in analyzer.analyze(text=text, language="en")]
recall_by_type(find_phi_presidio)
```

Presidio combines a named-entity model with pattern recognisers for emails, phone numbers and similar. Expect it to
find more names than the rules, and to need custom recognisers for local formats (record numbers, Indian postcodes,
Aadhaar or NHS numbers). Compare recall by identifier type and false positives on the same marked notes.

### Step 5

Write a short data-release checklist for sharing de-identified notes with a research partner.

Example data-release checklist for sharing de-identified notes with a research partner:

- [ ] Approval: ethics or data-governance approval and a data-sharing agreement that forbids re-identification,
      linkage and onward sharing, and sets retention and deletion.
- [ ] Legal basis checked under the applicable law (for example GDPR/UK GDPR, HIPAA, India's DPDP Act).
- [ ] Minimum necessary: only the notes and fields the project needs; free-text sections not needed are removed.
- [ ] Automated de-identification run, with recall measured on a marked sample from the same source and note types.
- [ ] Manual review of a random sample (and of all notes for rare conditions or unusual occupations, which can
      identify people), with a stop rule if any identifier is found.
- [ ] Dates shifted consistently per patient; ages over 89 grouped; small geographic areas generalised.
- [ ] Surrogate mapping and date shifts stored separately, encrypted, and not shared.
- [ ] Shared through a secure environment where possible, with access logging, rather than as a file copy.
- [ ] Release documented: what was shared, when, with whom, and the measured residual risk.

---

Lesson: [https://ramkumarmd.com/medical-ai/clinical-nlp/de-identifying-clinical-text](https://ramkumarmd.com/medical-ai/clinical-nlp/de-identifying-clinical-text). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.